In [1]:

!pip install --upgrade pip

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers
!pip install gradio
!pip install spacy
!pip install rapidfuzz
!pip install vaderSentiment
!pip install pandas
!pip install openai

!python -m spacy download en_core_web_md



Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 4.7 MB/s  0:00:07 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [2]:
import torch

ckpt = torch.load("bert.pth", map_location="cpu")
print(type(ckpt))


<class 'collections.OrderedDict'>


In [3]:
ckpt = torch.load("bert.pth", map_location="cpu")
print(list(ckpt.keys())[:40])


['bert.embeddings.word_embeddings.weight', 'bert.embeddings.position_embeddings.weight', 'bert.embeddings.token_type_embeddings.weight', 'bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.output.dense.weight', 'bert.encoder.layer.0.attention.output.dense.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.output.dense.weight', 'bert.encoder.layer.0.output.dense.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.

In [4]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel
import torch.nn.functional as F

class CustomTruthModel(nn.Module):
    def __init__(self, pretrained_state_dict=None):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")

        self.topic_embed = nn.Linear(60, 16)
        self.author_embed = nn.Embedding(22, 16)
        self.job_embed = nn.Embedding(22, 16)
        self.location_embed = nn.Embedding(28, 16)
        self.affiliation_embed = nn.Embedding(7, 16)

        self.author_feature_map = nn.Sequential(
            nn.Identity(), 
            nn.Linear(80, 128),
            nn.Identity(),
            nn.Identity(),
            nn.Linear(128, 128)
        )

        self.history_feature_map = nn.Sequential(
            nn.Linear(6, 128),
            nn.Identity(),
            nn.Identity(),
            nn.Linear(128, 128)
        )

        self.classifier = nn.Linear(1024, 5)

        if pretrained_state_dict:
            own_state = self.state_dict()
            for name, param in pretrained_state_dict.items():
                if name in own_state and own_state[name].shape == param.shape:
                    own_state[name].copy_(param)
            self.load_state_dict(own_state)

    def forward(self, input_ids, attention_mask,
                topic_vec, author_id, job_id, loc_id, aff_id,
                history_vec):

        bert_out = self.bert(input_ids=input_ids,
                             attention_mask=attention_mask)
        cls = bert_out.pooler_output

        t = self.topic_embed(topic_vec)
        a = self.author_embed(author_id)
        j = self.job_embed(job_id)
        l = self.location_embed(loc_id)
        f = self.affiliation_embed(aff_id)

        author_feat = torch.cat([t, a, j, l, f], dim=-1)
        author_feat = self.author_feature_map(author_feat)

        hist_feat = self.history_feature_map(history_vec)

        final = torch.cat([cls, author_feat, hist_feat], dim=-1)
        logits = self.classifier(final)
        return logits




/Users/ryanxavier/miniforge3/envs/rag_poc/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
!pip install spacy
!python -m spacy download en_core_web_md


  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl (33.5 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [6]:
import pandas as pd
from rapidfuzz import fuzz
import spacy

nlp = spacy.load("en_core_web_md")
statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}

def stat_counter(text):
    if not isinstance(text, str):
        return 0
    doc = nlp(text)
    return sum(ent.label_ in statistic_types for ent in doc.ents)

conservative_bigrams = pd.read_csv('top_conservative_bigrams.csv')['bigram'].tolist()
liberal_bigrams = pd.read_csv('top_liberal_bigrams.csv')['bigram'].tolist()

def match_counter(statement, bigram_list, threshold=70):
    words = [word.text.lower() for word in nlp(str(statement))]
    bigram_coll = [''.join(words[i:i+2]) for i in range(len(words)-1)]
    matches = 0
    for bigram in bigram_coll:
        for check in bigram_list:
            if fuzz.ratio(bigram, check) >= threshold:
                matches += 1
                break
    return matches

def political_bias_counts(statement):
    stat_count = stat_counter(statement)
    cons_bigram_count = match_counter(statement, conservative_bigrams)
    lib_bigram_count = match_counter(statement, liberal_bigrams)

    return {
        "stat_count": stat_count,
        "conservative_bigram_count": cons_bigram_count,
        "liberal_bigram_count": lib_bigram_count
    }


In [7]:
!pip install vaderSentiment

In [8]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def emotional_intensity_score(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    vs = analyzer.polarity_scores(text)
    return abs(vs['compound'])


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def spam_probability(text):
    inputs = spam_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )
    with torch.no_grad():
        outputs = spam_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        return probs[0, 1].item()  # probability that it's spam




In [10]:
import torch
import torch.nn.functional as F
from transformers import BertTokenizer


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

pretrained_state_dict = torch.load("bert.pth", map_location="cpu")

model = CustomTruthModel(pretrained_state_dict=pretrained_state_dict)
model.eval()  

def run_truth_model(article_text, 
                    topic_vec=None, author_id=None, job_id=None, loc_id=None, aff_id=None, history_vec=None):
    if topic_vec is None:
        topic_vec = torch.rand(1, 60)
    if author_id is None:
        author_id = torch.tensor([1])
    if job_id is None:
        job_id = torch.tensor([1])
    if loc_id is None:
        loc_id = torch.tensor([1])
    if aff_id is None:
        aff_id = torch.tensor([1])
    if history_vec is None:
        history_vec = torch.rand(1, 6)

    inputs = tokenizer(article_text, return_tensors="pt", truncation=True, padding="max_length", max_length=512)

    with torch.no_grad():
        logits = model(inputs['input_ids'], 
                       inputs['attention_mask'],
                       topic_vec, 
                       author_id, 
                       job_id, 
                       loc_id, 
                       aff_id, 
                       history_vec)
        probs = F.softmax(logits, dim=-1)
        pred_idx = torch.argmax(probs, dim=-1).item()
        class_labels = ["Mostly True", "True", "Half True", "Mostly False", "Pants on Fire"]
        pred_label = class_labels[pred_idx]

    return {
        "logits": logits,
        "probabilities": probs,
        "predicted_label": pred_label
    }




In [11]:
def analyze_for_genai(article_text):
    truth_result = run_truth_model(article_text)

    prob_tensor = truth_result["probabilities"].flatten().tolist()
    truth_probs = {
        "prob_mostly_true": prob_tensor[0],
        "prob_true": prob_tensor[1],
        "prob_half_true": prob_tensor[2],
        "prob_mostly_false": prob_tensor[3],
        "prob_pants_on_fire": prob_tensor[4],
    }

    bias_counts = political_bias_counts(article_text)
    bias_vector = [
        bias_counts["stat_count"],
        bias_counts["conservative_bigram_count"],
        bias_counts["liberal_bigram_count"]
    ]

    emotional_score = emotional_intensity_score(article_text)
    spam_score = spam_probability(article_text)

    feature_vector = list(truth_probs.values()) + bias_vector + [emotional_score, spam_score]

    return feature_vector


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "REDACTED"

In [ ]:
import os
from openai import OpenAI
import json
import gradio as gr
import re



def run_genai_with_vector(article_text, feature_vector):
    vector_str = ", ".join([f"{v:.3f}" if isinstance(v, float) else str(v) for v in feature_vector])

    prompt = f"""
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION
The feature vector contains the following predictive model outputs and auxiliary measures:
1-5: Probabilities for truthfulness classes from our custom BERT-based model:
     0 = Mostly True, 1 = True, 2 = Half True, 3 = Mostly False, 4 = Pants on Fire
6: Count of numeric/statistical entities detected in the text
7: Count of conservative bigram matches in the text
8: Count of liberal bigram matches in the text
9: Emotional intensity score (absolute VADER compound score)
10: Spam likelihood score (0–1, probability of being spam)

ANTI-BIAS CONSTRAINT
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

==========================
FACTUALITY FACTORS (6 TOTAL)
==========================

1. AUTHENTICITY
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM
- Definition: Presence of hyperbole, emotional language, exaggeration.
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.
- Output: score + 2 example phrases.

3. POLITICAL BIAS
- Definition: Degree to which the article leans left, center, or right.
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. TOXICITY
- Definition: Hostile, demeaning, or aggressive language.
- Scoring Recipe (1–10): Identify insults, threats, aggression, and target.
- Output: score + most toxic example phrase.

5. CONFIRMATION BIAS
- Definition: Selective presentation of information reinforcing a preferred conclusion.
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)
- Definition: Degree content maximizes clicks or engagement.
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.
- Output: score + 1–2 indicators of profit-driven framing.

OUTPUT FORMAT (STRICT JSON)
{{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "A well-detailed explanation explaining the final verdict and reconciling any discrepancies. Explain each factuality factor's score choice as well",
  "factor_scores": [
    {{"factor": "Authenticity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Sensationalism", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Political Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Toxicity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Confirmation Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Short-term Utility", "score": 1-10, "reasoning": "Brief evidence"}}
  ]
}}

ARTICLE TEXT:
\"\"\"
{article_text}
\"\"\"

PREDICTIVE MODEL FEATURE VECTOR:
[{vector_str}]
"""

    client = OpenAI(
        api_key = os.environ.get("REDACTED"), 
        base_url = "https://ellm.nrp-nautilus.io/v1"
    )

    completion = client.chat.completions.create(
        model="gemma3",
        messages=[
          
            {"role": "system", "content": "You are a helpful assistant that outputs STRICT JSON only."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1 
    )

    return completion.choices[0].message.content

def run_full_pipeline(article_text, progress=gr.Progress(track_tqdm=False)):
    if not article_text.strip():
        return "Error", [], "Please enter text.", {}

    try:
        progress(0.1, "Analyzing Features...")
        feature_vector = analyze_for_genai(article_text)

        progress(0.5, "Consulting LLM...")
        
        json_response_str = run_genai_with_vector(article_text, feature_vector)
        
        print(f"DEBUG: Raw Model Output:\n{json_response_str}") # Check your console if errors happen

        clean_str = re.sub(r"```json|```", "", json_response_str).strip()
        
        if "{" in clean_str:
            start = clean_str.find("{")
            end = clean_str.rfind("}") + 1
            clean_str = clean_str[start:end]

        data = json.loads(clean_str)

        veracity = data.get("veracity_label", "Unknown")
        scores_list = data.get("factor_scores", [])
        
        if not isinstance(scores_list, list):
             scores_list = []

        df_data = [[item["factor"], item["score"], item["reasoning"]] for item in scores_list]
        explanation = data.get("explanation_text", "")

        vector_display = {}
        if len(feature_vector) >= 10:
            vector_display = {
                "Probabilities (0-4)": feature_vector[0:5],
                "Numeric Entities": feature_vector[5],
                "Conservative Bigrams": feature_vector[6],
                "Liberal Bigrams": feature_vector[7],
                "Emotional Intensity": feature_vector[8],
                "Spam Score": feature_vector[9]
            }

        return veracity, df_data, explanation, vector_display

    except Exception as e:
        error_msg = f"⚠️ System Error: {str(e)}"
        print(error_msg)
        return "Error", [], error_msg, {}

with gr.Blocks(theme=gr.themes.Default(primary_hue="blue", secondary_hue="red")) as demo:
    gr.Markdown("# News Fact-Analysis")
    gr.Markdown("Paste an article below.")

    with gr.Row():
        with gr.Column(scale=1):
            article_input = gr.Textbox(
                label="Input Article", 
                placeholder="Paste text here...", 
                lines=15
            )
            submit_btn = gr.Button("Analyze Article", variant="primary")
        
        with gr.Column(scale=1):
            veracity_output = gr.Label(label="Veracity Verdict")
            explanation_output = gr.Markdown("### Analysis Summary\n*Run analysis to see details.*")
            scores_output = gr.Dataframe(
                headers=["Factor", "Score", "Reasoning"],
                datatype=["str", "number", "str"],
                label="Detailed Factor Scores",
                wrap=True
            )
            vector_output = gr.JSON(label="Underlying Model Vector")

    submit_btn.click(
        fn=run_full_pipeline,
        inputs=[article_input],
        outputs=[veracity_output, scores_output, explanation_output, vector_output]
    )

if __name__ == "__main__":
    demo.launch()



* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
print(run_truth_model("test"))
print(political_bias_counts("test"))


{'logits': tensor([[ 0.0468,  0.8338, -1.2021, -0.8570, -0.8792]]), 'probabilities': tensor([[0.2334, 0.5127, 0.0669, 0.0945, 0.0925]]), 'predicted_label': 'True'}
{'stat_count': 0, 'conservative_bigram_count': 0, 'liberal_bigram_count': 0}


DEBUG: Raw Model Output:
```json
{
  "veracity_label": "Pants on Fire",
  "explanation_text": "This article is demonstrably false and relies on extreme exaggeration and fabricated claims. The claims of Governor Newsom ordering citizens to wear full-body masks under threat of stoning or execution are entirely unsubstantiated and lack any credible sourcing. The predictive model scores, while leaning towards 'Mostly False' (0.250 for Mostly True, 0.477 for True), are not given undue weight as the textual evidence overwhelmingly points to fabrication. The article's sensationalism, political framing, and overall tone are indicative of disinformation.",
  "factor_scores": [
    {
      "factor": "Authenticity",
      "score": 1,
      "reasoning": "The article lacks any verifiable details, named sources beyond a fabricated quote from Governor Newsom, or timestamps. The claims are easily falsifiable and have no basis in reality."
    },
    {
      "factor": "Sensationalism",
      "score": 1